# Fine-tuning Multi-Modal Language Models Vertex AI

This notebook demonstrates how to fine-tune large language models using Vertex AI
and the Modelscope Swift framework. The process includes:

1. Setting up model and training configurations
2. Configuring Vertex AI resources
3. Fine-tuning the model
4. Evaluating the fine-tuning training process
5. Downloading and analyzing the fine-tuned model

## Key Components

- Model Configuration: Select and configure the model to be fine-tuned
- Vertex AI Setup: Configure AWS resources and training environment
- Training Process: Fine-tune the model using the SWIFT framework
- Evaluation: Analyze training metrics and model performance
- Model Export: Save and prepare the model for deployment

## Requirements

- Vertex AI access with appropriate permissions
- Training data in the correct format
- Sufficient GPU resources for training

In [15]:
import glob
SAMPLE_IMAGES = glob.glob("./data/electric_bill/images/*.png")

CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/4132743976048394240 current state:
JOB_STATE_RUNNING
CustomJob projects/73397202200/l

In [2]:
import sys
from utils.training_image import (
    get_sagemaker_distribution, 
    SageMakerDistribution, 
    get_python_version, 
    get_aws_account_id_for_region,
    is_docker_installed,
    is_docker_compose_installed,
    check_and_enable_docker_access_sagemaker_studio
)

In [1]:
import swift
print("swift module loaded from:", swift.__file__)
print("swift module path:", swift.__path__ if hasattr(swift, "__path__") else "N/A")
print("swift dir contents:", dir(swift))

swift module loaded from: d:\Anaconda\envs\ocr\Lib\site-packages\swift\__init__.py
swift module path: ['d:\\Anaconda\\envs\\ocr\\Lib\\site-packages\\swift']
swift dir contents: ['AdaLoraConfig', 'Adapter', 'AdapterConfig', 'AdapterModule', 'EvaluationStrategy', 'FSDPOption', 'HPSearchBackend', 'HubStrategy', 'IntervalStrategy', 'LoHaConfig', 'LoKrConfig', 'LoRA', 'LoRAConfig', 'LoftQConfig', 'LongLoRA', 'LongLoRAConfig', 'LongLoRAModelType', 'LoraConfig', 'OFTConfig', 'PeftConfig', 'PeftModel', 'PeftModelForCausalLM', 'PeftModelForSeq2SeqLM', 'PeftModelForSequenceClassification', 'PeftModelForTokenClassification', 'PrefixTuningConfig', 'Prompt', 'PromptConfig', 'PromptEncoderConfig', 'PromptLearningConfig', 'PromptModule', 'PromptTuningConfig', 'ResTuningConfig', 'SCETuning', 'SCETuningConfig', 'SWIFT_MAPPING', 'SchedulerType', 'Seq2SeqTrainer', 'Seq2SeqTrainingArguments', 'ShardedDDPOption', 'SideConfig', 'Swift', 'SwiftConfig', 'SwiftModel', 'SwiftOutput', 'SwiftTuners', 'Trainer', '

### Install Dependencies

In [19]:
%pip install -U --quiet requests beautifulsoup4 dataclasses google-cloud-aiplatform google-cloud-storage
%load_ext autoreload
%autoreload 2

Note: you may need to restart the kernel to use updated packages.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os
import json
import subprocess
import posixpath
from pathlib import Path
from typing import Optional
import time

from google.cloud import aiplatform, storage

In [4]:

from utils.config import ModelConfig
# Using Qwen2.5-VL-7B for its strong performance on vision-language tasks
# Can be a vision model that MS Swift supports: 
# https://github.com/modelscope/ms-swift/blob/main/docs/source_en/Instruction/Supported-models-and-datasets.md
model_config = ModelConfig(
    # model_type="qwen2_5_vl",
    # model_id="Qwen/Qwen2.5-VL-3B-Instruct"
    # Other models are
    # model_type = "deepseek_janus_pro",
    # model_id = "deepseek-ai/Janus-Pro-7B"

    # model_type = "qwen2_vl",
    # model_id = "Qwen/Qwen2-VL-2B-Instruct"

    model_type = "qwen2_5_vl",
    model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

    # model_type = "llama3_2_vision",
    # model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
)

print("✅ Configured model id.")

✅ Configured model id.


### Vertex AI Setup

In [5]:
import os, json
print(os.getcwd())
print(os.path.exists("gcs_config.json"))
with open("gcs_config.json") as f:
    print(json.load(f))

d:\Internship-Biwoco\Fine-tune\sample-for-multi-modal-document-to-json-with-sagemaker-ai
True
{'project_id': 'first-orc-chien', 'bucket_name': 'electric-bill-dataset-gcs', 'region': 'asia-southeast1', 'gcs_output_prefix': 'data/swift_dataset'}


In [7]:
# Initialize session and configure GCP resources for training
try:
    with open("gcs_config.json") as f:
        gcs_cfg = json.load(f)

    PROJECT_ID          = gcs_cfg["project_id"]
    default_bucket_name = gcs_cfg["bucket_name"]
    region              = gcs_cfg["region"]
    dataset_gcs_prefix  = gcs_cfg["gcs_output_prefix"]
    gcs_root_uri        = f"gs://{default_bucket_name}"
    dataset_gcs_uri     = f"{gcs_root_uri}/{dataset_gcs_prefix}"

    aiplatform.init(project=PROJECT_ID, location=region)

except Exception as e:
    raise Exception(f"Error setting up GCP session: {str(e)}")

print("✅ Initialized Vertex AI session...")
print(f"💾 Using dataset: {dataset_gcs_uri}")

✅ Initialized Vertex AI session...
💾 Using dataset: gs://electric-bill-dataset-gcs/data/swift_dataset


### Training Configuration

Next you will configure the training job. Here are the instance types available on Vertex AI:

| Machine Type | GPU | VRAM | Model | Note |
|---|---|---|---|---|
| a2-highgpu-1g | A100 40GB | 40GB | Qwen2.5-VL-7B | Recommended |
| a2-highgpu-2g | 2x A100 40GB | 80GB | Qwen2.5-VL-7B | Larger batch size |
| g2-standard-8 | L4 24GB | 24GB | Qwen2.5-VL-3B | Smaller model |

We can use preemptible instances for training to save up to 70% cost. Vertex AI handles restarting if interrupted.

In [8]:
# Vertex AI machine configuration
MACHINE_TYPE      = "a2-highgpu-1g"       # A100 40GB
ACCELERATOR_TYPE  = "NVIDIA_TESLA_A100"
ACCELERATOR_COUNT = 1
use_spot          = True                   # Use preemptible for cost saving (~70% cheaper)

In [9]:
# Setup training job parameters and checkpoint management
training_job_name_prefix = model_config.training_job_prefix(dataset_gcs_prefix)
print(f"Training job name prefix: {training_job_name_prefix}")

Training job name prefix: finetune-qwen2-5-vl-7b-instruct-data-sw


In [10]:
# Configure GCS checkpoint location
# Checkpointing is useful for preemptible instances to resume training after interruption
checkpoint_gcs_uri = posixpath.join(gcs_root_uri, training_job_name_prefix, "checkpoints")
print(f"Checkpoint GCS location: {checkpoint_gcs_uri}")

checkpoint_loc = None  # Set to checkpoint_gcs_uri to resume from previous run

Checkpoint GCS location: gs://electric-bill-dataset-gcs/finetune-qwen2-5-vl-7b-instruct-data-sw/checkpoints


In [ ]:
# To delete checkpoints:
# !gsutil -m rm -r {checkpoint_gcs_uri}

For the training you will need to set hyperparameters. We have already set sensible defaults for the parameters. You can overwrite any of them:

In [13]:
fine_tuning_kwargs = {
    "training_data_gcs":    dataset_gcs_uri,
    "checkpoint_loc":       checkpoint_loc,
    "model_type":           model_config.model_type,
    "model_id":             model_config.model_id,
    "train_data_path":      "conversations_train_swift_format.json",
    "validation_data_path": "conversations_dev_swift_format.json"
}

print(fine_tuning_kwargs)

{'training_data_gcs': 'gs://electric-bill-dataset-gcs/data/swift_dataset', 'checkpoint_loc': None, 'model_type': 'qwen2_5_vl', 'model_id': 'Qwen/Qwen2.5-VL-7B-Instruct', 'train_data_path': 'conversations_train_swift_format.json', 'validation_data_path': 'conversations_dev_swift_format.json'}


### Create Training Script

Vertex AI requires a standalone Python script to run inside the training container. (train.py)

## Submit Fine-Tuning Job to Vertex AI

This submits the training job to Vertex AI which will:
1. Upload the training script to GCS
2. Spin up an A100 instance
3. Run the training
4. Save model weights to GCS
5. Shut down the instance automatically

In [ ]:
from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID, 
    location=region,
    staging_bucket=f"gs://{default_bucket_name}"
)

job = aiplatform.CustomJob.from_local_script(
    display_name=training_job_name_prefix,
    script_path="train.py",
    container_uri="asia-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-3.py310:latest",
    requirements=[
        "ms-swift==3.2.2",
        "transformers==4.49.0",
        "qwen_vl_utils==0.0.11",
        "accelerate==1.1.0",
        "tensorboard",
        "tensorboardX",
        "decord",
        "av",
    ],
    machine_type="a2-highgpu-1g",
    accelerator_type="NVIDIA_TESLA_A100",
    accelerator_count=1,
    base_output_dir=f"gs://{default_bucket_name}/output",
    args=[
        "--model_type", fine_tuning_kwargs["model_type"],
        "--model_id", fine_tuning_kwargs["model_id"],
        "--training_data_gcs", fine_tuning_kwargs["training_data_gcs"],
    ]
)

print(f"🚀 Submitting job: {training_job_name_prefix}")
print(f"📊 View: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project={PROJECT_ID}")
job.run(sync=False)
print(f"✅ Job submitted!")

Training script copied to:
gs://electric-bill-dataset-gcs/aiplatform-2026-06-23-21:25:56.099-aiplatform_custom_trainer_script-0.1.tar.gz.
🚀 Submitting job: finetune-qwen2-5-vl-7b-instruct-data-sw
📊 View: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project=first-orc-chien
✅ Job submitted!
Creating CustomJob


CustomJob created. Resource name: projects/73397202200/locations/asia-southeast1/customJobs/6429579786007347200
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/73397202200/locations/asia-southeast1/customJobs/6429579786007347200')
View Custom Job:
https://console.cloud.google.com/agent-platform/locations/asia-southeast1/training/6429579786007347200?project=73397202200
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/6429579786007347200 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/6429579786007347200 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/6429579786007347200 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/6429579786007347200 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/6429579786007347200 current state:


In [ ]:
print(f"Region: {region}")
print(f"Job name: {job.display_name}")
print(f"Job state: {job.state}")

Region: asia-southeast1
Job name: finetune-qwen2-5-vl-7b-instruct-data-sw
Job state: 2


CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/locations/asia-southeast1/customJobs/7695091281298456576 current state:
JOB_STATE_PENDING
CustomJob projects/73397202200/l

## Download Model from GCS

After training completes, download the model weights from GCS to local directory.

- The checkpoint directory contains the actual LoRA adapter
- `adapter_model.safetensors` contains the actual weights

For inference you can either use the adapter together with the original model, or merge the adapter with the original model.

In [ ]:
# Download model from GCS
model_output_gcs  = f"gs://{default_bucket_name}/output"
model_weights_dir = "./models"
os.makedirs(model_weights_dir, exist_ok=True)

print(f"Downloading model from: {model_output_gcs}")
!gsutil -m cp -r {model_output_gcs} {model_weights_dir}/

model_dest_dir = f"{model_weights_dir}/output"
print(f"✅ Model downloaded to: {model_dest_dir}")

In [ ]:
# Inspect downloaded model structure
!du -ah --max-depth=5 {model_dest_dir}

In [ ]:
model_dir = model_dest_dir

from utils.helpers import find_latest_version_directory, find_best_model_checkpoint

latest_version   = find_latest_version_directory(model_dir)
latest_model_dir = os.path.join(model_dir, latest_version)
logging_file     = os.path.join(os.getcwd(), model_dir, latest_version, "logging.jsonl")

best_model_checkpoint = find_best_model_checkpoint(logging_file)
if best_model_checkpoint:
    best_model_checkpoint = best_model_checkpoint.replace("/gcs/output/", "")
    print(f"best model checkpoint: {best_model_checkpoint}")
else:
    print(
        "Best model checkpoint not found. Please search the logs manually to find the path that stores the best model checkpoint."
    )

## View Evaluation Metrics from fine-tuning run

Next you can look at the train & evaluation accuracy and loss.

In [ ]:
images_dir = os.path.join(latest_model_dir, "images")

In [ ]:
from IPython.display import Image
from IPython.display import display


def display_image(images_dir, image):
    image = Image(os.path.join(images_dir, image))
    display(image)

In [ ]:
display_image(images_dir, "train_token_acc.png")
display_image(images_dir, "train_loss.png")

In [ ]:
display_image(images_dir, "eval_token_acc.png")
display_image(images_dir, "eval_loss.png")

## Next Steps

1. Run inference on unseen data to evaluate the models real-world performance: [04_run_batch_inference.ipynb](04_run_batch_inference.ipynb) and then [05_evaluate_model.ipynb](05_evaluate_model.ipynb).
2. Deploy the model: [06_deploy_model_endpoint.ipynb](06_deploy_model_endpoint.ipynb)